# NB05: External validation strategies (Phase 5R)

LOSO + Cluster-KFold + Gridded-PT on the opx-liq track with pooled RMSE on
concatenated predictions as the primary metric. Features and hyperparameters
are dynamically loaded from the Phase 3R winning configuration; no feature
set is hard-coded.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)
from src.plot_style import load_winning_config
import warnings
warnings.filterwarnings('ignore')

import ast
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [2]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr


In [3]:
import ast
import json
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Dynamic setup: load data and canonical features from Phase 3R winner
df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')

config_3r = load_winning_config(RESULTS)
WIN_FEAT = config_3r['global_feature_set']
print(f'Phase 3R global winning feature set: {WIN_FEAT}')

# Validation uses the UNAUGMENTED feature set (no noise injection during CV)
X, feature_names = build_feature_matrix(df_liq, WIN_FEAT, use_liq=True)
groups_study = df_liq['Citation'].values
clusters = df_liq['chemical_cluster'].values
y_T = df_liq['T_C'].values
y_P = df_liq['P_kbar'].values

print(f'opx-liq rows: {len(df_liq)}, studies: {df_liq["Citation"].nunique()}, clusters: {df_liq["chemical_cluster"].nunique()}, X.shape: {X.shape}')

# Reconstruct best_params_by_model from the seed-42 rows of the canonical
# feature set in the multi-seed results CSV
ms = pd.read_csv(RESULTS / 'nb03_multi_seed_results.csv')
ms_canonical = ms[(ms['split_seed'] == 42) &
                  (ms['feature_set'] == WIN_FEAT) &
                  (ms['track'] == 'opx_liq')]

def _parse_params(s):
    if isinstance(s, dict):
        return s
    try:
        return ast.literal_eval(s)
    except Exception:
        try:
            return json.loads(s)
        except Exception:
            return {}

best_params_by_model = {}
for _, row in ms_canonical.iterrows():
    bp = _parse_params(row['best_params'])
    best_params_by_model.setdefault(row['model_name'], {})[row['target']] = bp

print('Best params reconstructed for models:', sorted(best_params_by_model.keys()))

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor

MODEL_CLASSES = {
    'RF': RandomForestRegressor,
    'ERT': ExtraTreesRegressor,
    'XGB': XGBRegressor,
    'GB': HistGradientBoostingRegressor,
}

def build_model(model_name, params):
    p = dict(params)
    if model_name == 'GB':
        return HistGradientBoostingRegressor(**p, random_state=SEED_MODEL)
    if model_name == 'XGB':
        return XGBRegressor(**p, random_state=SEED_MODEL, n_jobs=-1, verbosity=0)
    return MODEL_CLASSES[model_name](**p, random_state=SEED_MODEL, n_jobs=-1)

Phase 3R global winning feature set: pwlr
opx-liq rows: 600, studies: 93, clusters: 3, X.shape: (600, 72)
Best params reconstructed for models: ['ERT', 'GB', 'RF', 'XGB']


## Phase 5R.1: Three validation strategies

Runs LOSO, Cluster-KFold, and Gridded-PT on the canonical opx-liq feature
matrix with the frozen hyperparameters reconstructed above. The primary
metric is pooled RMSE across out-of-fold concatenated predictions; per-fold
RMSE is saved alongside for distributional diagnostics.


In [4]:
# Phase 5R.1: LOSO + Cluster-KFold + Gridded-PT execution
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

from src.evaluation import (
    loso_splits, cluster_kfold_splits, compute_metrics,
)
from tqdm.auto import tqdm

def pt_grid_labels(df, nT=3, nP=3):
    """Coarse P-T grid for spatial CV. Bin edges are quantile-based so
    fold sizes are roughly balanced."""
    T_bins = pd.qcut(df['T_C'], q=nT, labels=False, duplicates='drop')
    P_bins = pd.qcut(df['P_kbar'], q=nP, labels=False, duplicates='drop')
    return T_bins.astype(str) + '_' + P_bins.astype(str)


STRATEGY_SPLITTERS = {
    'LOSO': ('Citation', loso_splits),
    'Cluster-KFold': ('chemical_cluster', cluster_kfold_splits),
    'Gridded-PT': ('pt_grid', cluster_kfold_splits),
}

df_liq = df_liq.copy()
df_liq['pt_grid'] = pt_grid_labels(df_liq)

pooled_rows = []
per_fold_rows = []

for strategy, (group_col, split_fn) in tqdm(STRATEGY_SPLITTERS.items(),
                                             total=len(STRATEGY_SPLITTERS),
                                             desc='strategy'):
    groups = df_liq[group_col].values
    splits = split_fn(X, np.zeros(len(X)), groups)
    n_folds = len(splits)
    print(f'\n== {strategy} ({n_folds} folds, group={group_col}) ==')
    for model_name, params_for_model in tqdm(best_params_by_model.items(),
                                                total=len(best_params_by_model),
                                                desc=strategy, leave=False):
        for target_name, y in [('T_C', y_T), ('P_kbar', y_P)]:
            params = params_for_model.get(target_name, {})
            oof = np.full_like(y, np.nan, dtype=float)
            for f_idx, (tr, te) in enumerate(splits):
                est = build_model(model_name, params)
                est.fit(X[tr], y[tr])
                pred = predict_median(est, X[te])
                oof[te] = pred
                fold_rmse = float(np.sqrt(mean_squared_error(y[te], pred)))
                per_fold_rows.append({
                    'strategy': strategy, 'model_name': model_name,
                    'target': target_name, 'fold': f_idx,
                    'n_test': int(len(te)), 'rmse': fold_rmse,
                })
            mask = np.isfinite(oof)
            m = compute_metrics(y[mask], oof[mask])
            pooled_rows.append({
                'strategy': strategy, 'model_name': model_name,
                'target': target_name, 'feature_set': WIN_FEAT,
                'n_folds': n_folds,
                'rmse': m['rmse'], 'mae': m['mae'],
                'r2': m['r2'], 'bias': m['bias'], 'n': m['n'],
            })
            print(f'  {model_name:4s} {target_name:7s}  '
                  f'rmse={m["rmse"]:.2f}  r2={m["r2"]:+.3f}  n={m["n"]}')

pooled_df = pd.DataFrame(pooled_rows)
per_fold_df = pd.DataFrame(per_fold_rows)
pooled_df.to_csv(RESULTS / 'nb05_loso_pooled.csv', index=False)
per_fold_df.to_csv(RESULTS / 'nb05_per_fold_rmse.csv', index=False)
print(f'\nWrote {len(pooled_df)} pooled rows, {len(per_fold_df)} per-fold rows')


strategy:   0%|          | 0/3 [00:00<?, ?it/s]


== LOSO (93 folds, group=Citation) ==


LOSO:   0%|          | 0/4 [00:00<?, ?it/s]

  RF   T_C      rmse=74.65  r2=+0.821  n=600


  RF   P_kbar   rmse=5.93  r2=+0.599  n=600


...

  ERT  P_kbar   rmse=8.43  r2=+0.189  n=600


  XGB  T_C      rmse=107.72  r2=+0.627  n=600


  XGB  P_kbar   rmse=7.75  r2=+0.315  n=600


  GB   T_C      rmse=97.70  r2=+0.693  n=600


  GB   P_kbar   rmse=7.75  r2=+0.315  n=600

Wrote 24 pooled rows, 840 per-fold rows


In [5]:
# Phase 5R.2: verification
pooled_df = pd.read_csv(RESULTS / 'nb05_loso_pooled.csv')
per_fold_df = pd.read_csv(RESULTS / 'nb05_per_fold_rmse.csv')
assert set(pooled_df['strategy'].unique()) == {'LOSO', 'Cluster-KFold', 'Gridded-PT'}
assert pooled_df['model_name'].nunique() == 4
assert pooled_df['target'].nunique() == 2
assert (pooled_df['n_folds'] >= 2).all()
assert set(per_fold_df['strategy'].unique()) == {'LOSO', 'Cluster-KFold', 'Gridded-PT'}

print('=== PHASE 5R COMPLETE ===')
print(f'  {len(pooled_df)} pooled rows across 3 strategies x 4 models x 2 targets')
print(f'  {len(per_fold_df)} per-fold rows')


=== PHASE 5R COMPLETE ===
  24 pooled rows across 3 strategies x 4 models x 2 targets
  840 per-fold rows
